# Explore an ensemblr ensemble

A minimal starting point for your own analysis of an ensemble made by `ensemblr-generate`.
It loads the structures, runs PCA and clustering, and then shows how to pick structures
(by PC value, near a point, per cluster), write them out, add your own per-structure
properties and use your own clustering.

**Edit the Settings cell, then run top to bottom.** The first run builds the analysis and saves it to
`RUN_DIR`; later runs reopen it in seconds.

The objects you work with:

| Object | What it is |
|---|---|
| `ens` | the `Ensemble`: everything below hangs off it |
| `ens.df` | the table: one row per structure (indexed by PDB file name), columns `pLDDT`, `PC1…`, `cluster`, plus anything you add |
| `ens.universe` | all structures as one aligned MDAnalysis Universe; frame *i* is row *i* of `ens.df` |
| `ens.pca` | the PCA model (`p_components`, `mean`, `variance`) |

Any rows of `ens.df` you pick (with pandas, or `select` / `nearest` / `representatives`) can be passed
straight to `ens.write(...)` or `ens.paths(...)`.

For umbrella sampling along a path or a 2D grid, use the umbrella-sampling template notebooks
(`ensemblr/templates/`) instead.

In [ ]:
# ── Settings: edit these ─────────────────────────────────────────────────────

# The generator's output directory: the folder holding structures/ and plddt_dict.json
# (usually <target>/synthetic_ensemble/).
ENSEMBLE_DIR = '/path/to/synthetic_ensemble/'

# Where this notebook saves its analysis and reopens it from next time.
# None = <ENSEMBLE_DIR>/explore/. You can also point it at a run folder of the umbrella-sampling
# notebooks (e.g. <ENSEMBLE_DIR>/run_01/) to reuse its PCA and clusters; the last cell saves back into it.
RUN_DIR = None

# Set True to redo the analysis from scratch (needed after changing N_PCS or MIN_PLDDT, or after the
# generator has added structures). False reopens the saved analysis if there is one.
REBUILD = False

N_PCS = 3          # principal components to keep (at least 2)
MIN_PLDDT = 0      # drop structures below this pLDDT (the generator has already filtered on quality)

In [ ]:
import os
import json
import shutil
import warnings
warnings.filterwarnings('ignore')
os.environ.setdefault('JAX_PLATFORMS', 'cpu')   # ensemblr imports JAX; keep it off the GPU here

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ensemblr import Ensemble
warnings.filterwarnings('ignore')               # again: MDAnalysis re-enables its deprecation warnings on import

# ── Check the settings: stops here with a message if something is wrong ──────
ENSEMBLE_DIR = os.path.abspath(os.path.expanduser(ENSEMBLE_DIR)) + '/'
STRUCTURES = ENSEMBLE_DIR + 'structures/'
QUALITY_JSON = ENSEMBLE_DIR + 'plddt_dict.json'

if not os.path.isdir(ENSEMBLE_DIR):
    raise FileNotFoundError(f'ENSEMBLE_DIR does not exist: {ENSEMBLE_DIR}')
if not (os.path.isdir(STRUCTURES) and os.path.exists(QUALITY_JSON)):
    hint = (f' Did you mean {ENSEMBLE_DIR}synthetic_ensemble/ ?'
            if os.path.isdir(ENSEMBLE_DIR + 'synthetic_ensemble') else '')
    raise FileNotFoundError(f'{ENSEMBLE_DIR} is not an ensemblr-generate output folder '
                            f'(it needs structures/ and plddt_dict.json).{hint}')
if N_PCS < 2:
    raise ValueError('N_PCS must be at least 2 (the plots below use PC1 and PC2)')

RUN_DIR = os.path.abspath(os.path.expanduser(RUN_DIR or ENSEMBLE_DIR + 'explore')) + '/'
OUT = RUN_DIR + 'picked/'                       # where the examples below write structures
os.makedirs(OUT, exist_ok=True)
if not os.access(RUN_DIR, os.W_OK):
    raise PermissionError(f'cannot write to RUN_DIR {RUN_DIR}; set RUN_DIR to a folder of your own')

if not os.path.exists(ENSEMBLE_DIR + 'structures.xtc'):
    print(f'Tip: writing structures out is much faster with the run trajectory. Make it once with:\n'
          f'    ensemblr-xtc {ENSEMBLE_DIR}\n')
print(f'Ensemble : {ENSEMBLE_DIR}\nAnalysis : {RUN_DIR}\nOutput   : {OUT}')

## 1. Load (or build) the ensemble

If `RUN_DIR` already holds a saved analysis (and `REBUILD` is False), it is reopened in seconds.
Otherwise the structures are loaded, superposed and projected onto their principal components, then
clustered with HDBSCAN and saved. That takes a few minutes for ~10,000 structures, and writes
`ensemble.pdb` (a few hundred MB per thousand structures) to `RUN_DIR`.

In [ ]:
saved = os.path.join(RUN_DIR, 'ensemble_df.parquet')

if os.path.exists(saved) and not REBUILD:
    ens = Ensemble.from_run(RUN_DIR, structure_directory=STRUCTURES)
    if os.path.getmtime(QUALITY_JSON) > os.path.getmtime(saved):
        print('\nWARNING: the generator has written structures since this analysis was saved, so new '
              'structures are missing from it. Set REBUILD = True to include them.')
else:
    if REBUILD and os.path.exists(saved):
        os.remove(saved)        # so an interrupted rebuild is never mistaken for a finished one
    ens = Ensemble(STRUCTURES, thresh_pLDDT=MIN_PLDDT)

    # Quality score (pLDDT, or ipTM for complexes) of every accepted structure, from the generator
    with open(QUALITY_JSON) as fh:
        quality = {os.path.basename(k): v for k, v in json.load(fh).items()}
    ens.load(plddt_dict=quality)

    # Atoms used to superpose the structures and for PCA. The generator's choice (the secondary-
    # structure core, Cα) is a good default. To use your own, set any MDAnalysis selection string
    # here instead, e.g. ens.rmsd_selection = 'name CA and resid 20-300'.
    if os.path.exists(ENSEMBLE_DIR + 'pca_selection.txt'):
        with open(ENSEMBLE_DIR + 'pca_selection.txt') as fh:
            ens.rmsd_selection = fh.read().strip()
    else:
        print('No pca_selection.txt from the generator; using all Cα atoms.')
        ens.rmsd_selection = 'name CA'

    ens.run_pca(n_components=N_PCS, output_dir=RUN_DIR)   # adds PC1..PCn to ens.df
    ens.cluster()                                         # adds 'cluster' (HDBSCAN on the PCs; -1 = noise)
    ens.save(RUN_DIR)

PCS = [c for c in ens.df.columns if c.startswith('PC') and c[2:].isdigit()]   # ['PC1', 'PC2', ...]
if len(PCS) != N_PCS:
    print(f'\nNote: the saved analysis has {len(PCS)} PCs, not N_PCS = {N_PCS}. Set REBUILD = True to redo it.')

n_clusters = len(set(ens.df['cluster']) - {-1}) if 'cluster' in ens.df else 0
print(f'\n{len(ens.df)} structures | PCs: {", ".join(PCS)} | HDBSCAN clusters: {n_clusters}')
print(f'Columns of ens.df: {", ".join(ens.df.columns)}')

## 2. The table

Everything known about each structure is a column of `ens.df`, a pandas DataFrame indexed by file
name. Filter, sort and plot it like any DataFrame.

In [ ]:
ens.df.head()

In [ ]:
ens.df.describe()

## 3. Look at PC space

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharex=True, sharey=True)

axes[0].hexbin(ens.df['PC1'], ens.df['PC2'], gridsize=50, mincnt=1, bins='log', cmap='viridis')
axes[0].set_title('Density')

for label, group in ens.df.groupby('cluster'):
    axes[1].scatter(group['PC1'], group['PC2'], s=2, label='noise' if label == -1 else f'cluster {label}',
                    color='lightgrey' if label == -1 else None)
if ens.df['cluster'].nunique() <= 12:
    axes[1].legend(markerscale=5, fontsize=8)
axes[1].set_title('HDBSCAN clusters (grey = noise)')

for ax in axes:
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
plt.tight_layout()

## 4. Pick structures

Each of these returns a DataFrame of rows from `ens.df`.

In [ ]:
# The 10 structures with the highest and the lowest PC1 (plain pandas)
pc1_high = ens.df.nlargest(10, 'PC1')
pc1_low = ens.df.nsmallest(10, 'PC1')

# A box in PC space: ranges as (low, high), None = no limit
box = ens.select(PC1=(None, 0), PC2=(-2, 2))

# Any condition, as a pandas query string
confident_high = ens.select('PC1 > 0 and pLDDT > 85')

# The 5 structures closest to a point (adds a 'distance' column).
# Use scale=True when mixing columns with different units, e.g. a PC and a distance in Å.
near = ens.nearest(PC1=0.0, PC2=0.0, n=5)

# One structure per bin along PC1 (closest to each bin centre): a sweep along the first mode
pc1_sweep = ens.representatives('PC1', n_bins=10)

# One structure per cluster (the member closest to the cluster centroid in PC space).
# Empty if HDBSCAN found no clusters (everything labelled -1, noise): then try ens.cluster(min_samples_frac=0.02),
# or your own clustering (section 7).
cluster_reps = ens.representatives('cluster')

print(f'box: {len(box)} structures, PC1 > 0 and pLDDT > 85: {len(confident_high)}, '
      f'clusters: {len(cluster_reps)}')
cluster_reps

## 5. Write them out

Structures are written superposed on the first structure. The output type follows the name:

- `name.xtc` writes a trajectory, plus `name.pdb` (topology) and `name_index.csv` (frame → structure, with all columns)
- `name.pdb` writes one multi-model PDB
- `some_dir/` writes one PDB per structure, keeping the original file names

`selection=` limits the atoms written, e.g. `'protein and name CA'`. `ens.paths(...)` gives the
original PDB files instead.

In [ ]:
ens.write(pc1_high, OUT + 'pc1_high.xtc')        # view with: vmd pc1_high.pdb pc1_high.xtc
ens.write(pc1_low, OUT + 'pc1_low.pdb')
ens.write(pc1_sweep, OUT + 'pc1_sweep.xtc')      # plays like a morph along PC1
if len(cluster_reps):
    ens.write(cluster_reps, OUT + 'cluster_reps/')

print(f'\nTo look at them:\n  vmd {OUT}pc1_high.pdb {OUT}pc1_high.xtc\n  pymol {OUT}pc1_low.pdb')
ens.paths(near)                                  # absolute paths of the original PDBs

## 6. Add your own per-structure property

Loop over `ens.universe` (one frame per row of `ens.df`) and store the result as a new column. It can
then be plotted, selected on and clustered like the PCs. Building the universe takes a minute or two
the first time for a large ensemble.

The example computes the radius of gyration and the N-to-C-terminal Cα distance. Replace them with
whatever you need (distances between residues, angles, contacts...).

In [ ]:
u = ens.universe
protein = u.select_atoms('protein')
ca = u.select_atoms('name CA')
n_term, c_term = ca[0], ca[-1]

rg, ends = [], []
for ts in u.trajectory:
    rg.append(protein.radius_of_gyration())
    ends.append(np.linalg.norm(n_term.position - c_term.position))

ens.df['Rg'] = rg
ens.df['NC_distance'] = ends

ens.df.plot.scatter('PC1', 'Rg', s=2, c='NC_distance', cmap='viridis', figsize=(5, 4));

In [ ]:
# New columns work everywhere: e.g. the most compact structures with PC1 > 0
compact = ens.select(PC1=(0, None)).nsmallest(5, 'Rg')
compact[['PC1', 'PC2', 'Rg', 'NC_distance']]

## 7. Your own clustering

Any clustering that gives one label per row works: store the labels as a column, then pick a
representative per cluster, or select a whole cluster. Here k-means on the first three PCs.

In [ ]:
from sklearn.cluster import KMeans

features = PCS[:3]              # columns to cluster on; any columns work, e.g. ['PC1', 'Rg'] (scale them if units differ)
N_CLUSTERS = 5
ens.df['kmeans'] = KMeans(n_clusters=min(N_CLUSTERS, len(ens.df)), n_init=10,
                          random_state=0).fit_predict(ens.df[features])

kmeans_reps = ens.representatives('kmeans', columns=features)
ens.write(kmeans_reps, OUT + 'kmeans_reps.pdb')
ens.write(ens.select(kmeans=0), OUT + 'kmeans_0.xtc')   # a whole cluster as a trajectory

ax = ens.df.plot.scatter('PC1', 'PC2', c='kmeans', cmap='tab10', s=2, colorbar=False, figsize=(5, 4))
ax.scatter(kmeans_reps['PC1'], kmeans_reps['PC2'], marker='x', s=80, c='k');
kmeans_reps[features + ['group', 'n_members']]

## 8. Save

Saves the table with your new columns, so the next `Ensemble.from_run(RUN_DIR)` has them.

In [ ]:
ens.save(RUN_DIR)
print(f'Saved to {RUN_DIR}. Next time, this notebook reopens it in seconds.')